# Student Dropout Prediction and Explainable Assistant

**Kaggle edition.** This notebook goes from the raw CSV to a saved, explainable
XGBoost model and an interactive risk assistant.

## How to run

1. Add `dataset_outliers_removed_balanced.csv` as a Kaggle dataset.
2. Open this notebook in Kaggle.
3. Click **Run all**. A CPU is enough; no GPU is required.
4. Use the 14-field widget form near the bottom to get a prediction and
   explanation.
5. Download `student_dropout_project.zip` from `/kaggle/working`.

## What you get

- A trained XGBoost dropout model
- A saved preprocessing object, feature list, threshold, category map, metadata
- Evaluation report, XGBoost importance, global SHAP, category contribution
- `src/` modules for later local use
- A Streamlit app for the local shift
- A 14-field ipywidgets chatbot that works inside Kaggle

All preprocessing is fitted on the training split only. The test split is used
only for final evaluation. `RANDOM_STATE = 42`.

## Step 0 - Setup

This cell imports everything once, creates the project folders, finds the CSV,
and moves into the project directory. If the dataset is not found, add it as a
Kaggle dataset and rerun this cell.

In [ ]:
import os, sys, json, glob, shutil, zipfile, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import xgboost as xgb
import ipywidgets as widgets
from IPython.display import display, HTML

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
SOURCE_DIR = Path.cwd()
if Path("/kaggle/working").exists():
    PROJECT_DIR = Path("/kaggle/working/student_dropout_project")
else:
    PROJECT_DIR = SOURCE_DIR / "student_dropout_project"

DATA_DIR = PROJECT_DIR / "data"
MODELS_DIR = PROJECT_DIR / "models"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
SRC_DIR = PROJECT_DIR / "src"
APP_DIR = PROJECT_DIR / "app"
for folder in [DATA_DIR, MODELS_DIR, OUTPUTS_DIR, SRC_DIR, APP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "__init__.py").write_text(
    '"""Student dropout prediction package."""\n', encoding="utf-8"
)


def find_dataset():
    roots = [Path("/kaggle/input"), SOURCE_DIR, Path("/kaggle/working")]
    hits = []
    for root in roots:
        if root.exists():
            hits.extend(root.rglob("dataset_outliers_removed_balanced.csv"))
    hits = sorted(set(hits))
    if not hits:
        raise FileNotFoundError(
            "Add dataset_outliers_removed_balanced.csv as a Kaggle dataset, "
            "then rerun this cell."
        )
    return hits[0]


DATASET_PATH = find_dataset()
LOCAL_DATA_PATH = DATA_DIR / DATASET_PATH.name
shutil.copy2(DATASET_PATH, LOCAL_DATA_PATH)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

RISK_BANDS = {
    "Low": [0.00, 0.35],
    "Medium": [0.35, 0.65],
    "High": [0.65, 0.85],
    "Critical": [0.85, 1.01],
}

print("Dataset:", DATASET_PATH)
print("Project:", PROJECT_DIR)
print("Python:", sys.version.split()[0])
print("xgboost:", xgb.__version__, "| shap:", shap.__version__)

## Step 1 - Load and inspect the data

We check the shape, columns, missing values, duplicates, and target values. The
target is cleaned to lowercase, then three target columns are created: the main
binary `is_dropout`, the optional three-class `target_class`, and `is_graduate`.

In [ ]:
df = pd.read_csv(LOCAL_DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
display(df.head())
print("\nMissing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\nTarget distribution:")
print(df["Target"].value_counts(dropna=False))

df["Target"] = df["Target"].astype(str).str.strip().str.lower()
valid_targets = {"graduate", "dropout", "enrolled"}
unexpected = sorted(set(df["Target"].unique()) - valid_targets)
assert not unexpected, f"Unexpected target values: {unexpected}"

df = (
    df.replace([np.inf, -np.inf], np.nan)
      .drop_duplicates()
      .reset_index(drop=True)
)

df["is_dropout"] = (df["Target"] == "dropout").astype(int)
df["target_class"] = df["Target"].map(
    {"dropout": 0, "enrolled": 1, "graduate": 2}
)
df["is_graduate"] = (df["Target"] == "graduate").astype(int)

print("\nCleaned shape:", df.shape)
print("\nBinary target:")
print(df["is_dropout"].value_counts().rename({0: "Not dropout", 1: "Dropout"}))
print("\nThree-class target:")
print(
    df["target_class"]
      .value_counts()
      .sort_index()
      .rename({0: "Dropout", 1: "Enrolled", 2: "Graduate"})
)

## Step 2 - Feature engineering module

The CSV uses numeric codes for categories such as `Course` and `Application
mode`, so those are kept separate from genuinely numeric columns. This module
contains the feature groups, the semester-level feature formulas, the category
mapping, and human-readable labels. It is written to `src/` so the same logic is
reused by the notebook, the prediction module, and the local app.

In [ ]:
%%writefile src/feature_engineering.py
"""Feature engineering, feature grouping, and human-readable labels."""

from __future__ import annotations

import numpy as np
import pandas as pd

TARGET_COLS = ["Target", "is_dropout", "target_class", "is_graduate"]

CATEGORICAL_CODE_COLS = [
    "Marital status",
    "Application mode",
    "Application order",
    "Course",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
]

BINARY_FLAG_COLS = [
    "Daytime/evening attendance",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International",
]

# The simplified form asks only for the fields that carry the strongest,
# practical signal. The trained model still uses its full feature set; omitted
# background fields are filled with typical training values and are not shown
# in the key-factor list.
APP_SECTIONS = {
    "Student details": [
        "Course",
        "Age at enrollment",
        "Gender",
        "Displaced",
    ],
    "Academic performance": [
        "Admission grade",
        "Curricular units 1st sem (enrolled)",
        "Curricular units 1st sem (approved)",
        "Curricular units 1st sem (grade)",
        "Curricular units 2nd sem (enrolled)",
        "Curricular units 2nd sem (approved)",
        "Curricular units 2nd sem (grade)",
    ],
    "Financial support": [
        "Tuition fees up to date",
        "Debtor",
        "Scholarship holder",
    ],
}
APP_INPUT_FIELDS = [field for fields in APP_SECTIONS.values() for field in fields]

INT_INPUT_FIELDS = {
    "Age at enrollment",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (approved)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (approved)",
}

# Raw UCI course codes used by this dataset. The app shows the names instead
# of asking people to remember the numeric codes.
COURSE_LABELS = {
    33: "Biofuel Production Technologies",
    171: "Animation and Multimedia Design",
    8014: "Social Service (evening attendance)",
    9003: "Agronomy",
    9070: "Communication Design",
    9085: "Veterinary Nursing",
    9119: "Informatics Engineering",
    9130: "Equiniculture",
    9147: "Management",
    9238: "Social Service",
    9254: "Tourism",
    9500: "Nursing",
    9556: "Oral Hygiene",
    9670: "Advertising and Marketing Management",
    9773: "Journalism and Communication",
    9853: "Basic Education",
    9991: "Management (evening attendance)",
}

MACRO_COLS = ["Unemployment rate", "Inflation rate", "GDP"]

FINANCIAL_FEATURES = {"Debtor", "Tuition fees up to date", "Scholarship holder"}

ACADEMIC_FEATURES = {
    "Previous qualification",
    "Previous qualification (grade)",
    "Admission grade",
    "effective_avg_grade_sem1",
    "effective_avg_grade_sem2",
    "overall_average_grade",
    "grade_difference_sem2_minus_sem1",
}

DEMOGRAPHIC_FEATURES = {
    "Marital status",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Gender",
    "Age at enrollment",
    "International",
}

INSTITUTIONAL_FEATURES = {
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
}

MACRO_FEATURES = set(MACRO_COLS)

# Features that are fully determined by the simplified form. Explanations are
# filtered to this set so the app never presents a background field that the
# user did not enter as a key factor.
APP_EXPLANATION_FEATURES = set(APP_INPUT_FIELDS) | {
    "total_enrolled_units",
    "total_approved_units",
    "failed_units_sem1",
    "failed_units_sem2",
    "total_failed_units",
    "approval_rate_sem1",
    "approval_rate_sem2",
    "overall_approval_rate",
    "effective_avg_grade_sem1",
    "effective_avg_grade_sem2",
    "overall_average_grade",
    "grade_difference_sem2_minus_sem1",
    "approved_difference_sem2_minus_sem1",
    "approval_rate_difference_sem2_minus_sem1",
    "failed_difference_sem2_minus_sem1",
    "no_approved_sem1",
    "no_approved_sem2",
    "no_approved_both_semesters",
    "low_approval_sem1",
    "low_approval_sem2",
    "low_approval_both_semesters",
}

DISPLAY_NAMES = {
    "Marital status": "Marital status",
    "Application mode": "Application mode",
    "Application order": "Application order",
    "Course": "Course",
    "Daytime/evening attendance": "Daytime/evening attendance",
    "Previous qualification": "Previous qualification",
    "Previous qualification (grade)": "Previous qualification grade",
    "Nacionality": "Nationality",
    "Mother's qualification": "Mother's qualification",
    "Father's qualification": "Father's qualification",
    "Mother's occupation": "Mother's occupation",
    "Father's occupation": "Father's occupation",
    "Admission grade": "Admission grade",
    "Displaced": "Displaced student",
    "Educational special needs": "Educational special needs",
    "Debtor": "Outstanding debt",
    "Tuition fees up to date": "Tuition fees up to date",
    "Gender": "Gender",
    "Scholarship holder": "Scholarship holder",
    "Age at enrollment": "Age at enrollment",
    "International": "International student",
    "Curricular units 1st sem (credited)": "Semester 1 credited units",
    "Curricular units 1st sem (enrolled)": "Semester 1 enrolled units",
    "Curricular units 1st sem (evaluations)": "Semester 1 evaluations",
    "Curricular units 1st sem (approved)": "Semester 1 approved units",
    "Curricular units 1st sem (grade)": "Semester 1 grade",
    "Curricular units 1st sem (without evaluations)": "Semester 1 units without evaluations",
    "Curricular units 2nd sem (credited)": "Semester 2 credited units",
    "Curricular units 2nd sem (enrolled)": "Semester 2 enrolled units",
    "Curricular units 2nd sem (evaluations)": "Semester 2 evaluations",
    "Curricular units 2nd sem (approved)": "Semester 2 approved units",
    "Curricular units 2nd sem (grade)": "Semester 2 grade",
    "Curricular units 2nd sem (without evaluations)": "Semester 2 units without evaluations",
    "Unemployment rate": "Unemployment rate",
    "Inflation rate": "Inflation rate",
    "GDP": "GDP",
    "total_enrolled_units": "Total enrolled units",
    "total_approved_units": "Total approved units",
    "total_credited_units": "Total credited units",
    "total_evaluations": "Total evaluations",
    "total_without_evaluations": "Total units without evaluations",
    "failed_units_sem1": "Failed units in semester 1",
    "failed_units_sem2": "Failed units in semester 2",
    "total_failed_units": "Total failed units",
    "approval_rate_sem1": "Semester 1 approval rate",
    "approval_rate_sem2": "Semester 2 approval rate",
    "overall_approval_rate": "Overall approval rate",
    "effective_avg_grade_sem1": "Semester 1 effective grade",
    "effective_avg_grade_sem2": "Semester 2 effective grade",
    "overall_average_grade": "Overall average grade",
    "grade_difference_sem2_minus_sem1": "Grade change from semester 1 to 2",
    "approved_difference_sem2_minus_sem1": "Approved-unit change from semester 1 to 2",
    "approval_rate_difference_sem2_minus_sem1": "Approval-rate change from semester 1 to 2",
    "failed_difference_sem2_minus_sem1": "Failed-unit change from semester 1 to 2",
    "no_approved_sem1": "No approved units in semester 1",
    "no_approved_sem2": "No approved units in semester 2",
    "no_approved_both_semesters": "No approved units in both semesters",
    "low_approval_sem1": "Low approval rate in semester 1",
    "low_approval_sem2": "Low approval rate in semester 2",
    "low_approval_both_semesters": "Low approval rate in both semesters",
    "evaluation_to_enrolled_ratio_sem1": "Semester 1 evaluation-to-enrollment ratio",
    "evaluation_to_enrolled_ratio_sem2": "Semester 2 evaluation-to-enrollment ratio",
    "overall_evaluation_to_enrolled_ratio": "Overall evaluation-to-enrollment ratio",
}


def safe_div(numerator, denominator):
    """Divide two series and return 0 wherever the denominator is zero."""
    num = numerator if isinstance(numerator, pd.Series) else pd.Series(numerator)
    den = denominator if isinstance(denominator, pd.Series) else pd.Series(denominator)
    den = den.replace(0, np.nan)
    result = num / den
    return result.replace([np.inf, -np.inf], np.nan).fillna(0.0)


def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """Create the semester-level academic features used by the model."""
    d = data.copy()

    s1_cred = d["Curricular units 1st sem (credited)"]
    s1_enr = d["Curricular units 1st sem (enrolled)"]
    s1_eval = d["Curricular units 1st sem (evaluations)"]
    s1_app = d["Curricular units 1st sem (approved)"]
    s1_grade = d["Curricular units 1st sem (grade)"]
    s1_noeval = d["Curricular units 1st sem (without evaluations)"]

    s2_cred = d["Curricular units 2nd sem (credited)"]
    s2_enr = d["Curricular units 2nd sem (enrolled)"]
    s2_eval = d["Curricular units 2nd sem (evaluations)"]
    s2_app = d["Curricular units 2nd sem (approved)"]
    s2_grade = d["Curricular units 2nd sem (grade)"]
    s2_noeval = d["Curricular units 2nd sem (without evaluations)"]

    d["total_enrolled_units"] = s1_enr + s2_enr
    d["total_approved_units"] = s1_app + s2_app
    d["total_credited_units"] = s1_cred + s2_cred
    d["total_evaluations"] = s1_eval + s2_eval
    d["total_without_evaluations"] = s1_noeval + s2_noeval

    d["failed_units_sem1"] = (s1_enr - s1_app).clip(lower=0)
    d["failed_units_sem2"] = (s2_enr - s2_app).clip(lower=0)
    d["total_failed_units"] = (
        d["total_enrolled_units"] - d["total_approved_units"]
    ).clip(lower=0)

    d["approval_rate_sem1"] = safe_div(s1_app, s1_enr).clip(0, 1)
    d["approval_rate_sem2"] = safe_div(s2_app, s2_enr).clip(0, 1)
    d["overall_approval_rate"] = safe_div(
        d["total_approved_units"], d["total_enrolled_units"]
    ).clip(0, 1)

    d["effective_avg_grade_sem1"] = np.where(s1_app > 0, s1_grade, 0.0)
    d["effective_avg_grade_sem2"] = np.where(s2_app > 0, s2_grade, 0.0)
    d["overall_average_grade"] = safe_div(
        d["effective_avg_grade_sem1"] * s1_app
        + d["effective_avg_grade_sem2"] * s2_app,
        d["total_approved_units"],
    )

    d["grade_difference_sem2_minus_sem1"] = (
        d["effective_avg_grade_sem2"] - d["effective_avg_grade_sem1"]
    )
    d["approved_difference_sem2_minus_sem1"] = s2_app - s1_app
    d["approval_rate_difference_sem2_minus_sem1"] = (
        d["approval_rate_sem2"] - d["approval_rate_sem1"]
    )
    d["failed_difference_sem2_minus_sem1"] = (
        d["failed_units_sem2"] - d["failed_units_sem1"]
    )

    d["no_approved_sem1"] = (s1_app == 0).astype(int)
    d["no_approved_sem2"] = (s2_app == 0).astype(int)
    d["no_approved_both_semesters"] = (
        (s1_app == 0) & (s2_app == 0)
    ).astype(int)
    d["low_approval_sem1"] = (d["approval_rate_sem1"] < 0.5).astype(int)
    d["low_approval_sem2"] = (d["approval_rate_sem2"] < 0.5).astype(int)
    d["low_approval_both_semesters"] = (
        (d["approval_rate_sem1"] < 0.5) & (d["approval_rate_sem2"] < 0.5)
    ).astype(int)

    d["evaluation_to_enrolled_ratio_sem1"] = safe_div(s1_eval, s1_enr)
    d["evaluation_to_enrolled_ratio_sem2"] = safe_div(s2_eval, s2_enr)
    d["overall_evaluation_to_enrolled_ratio"] = safe_div(
        d["total_evaluations"], d["total_enrolled_units"]
    )

    return d.replace([np.inf, -np.inf], np.nan)


def get_feature_columns(data: pd.DataFrame) -> list[str]:
    """Return every non-target column after feature engineering."""
    return [c for c in data.columns if c not in TARGET_COLS]


def build_category_map(feature_columns) -> dict[str, str]:
    """Map each model feature to a human-readable explanation category."""
    category_map = {}
    for feature in feature_columns:
        if feature in FINANCIAL_FEATURES:
            category = "Financial"
        elif feature in ACADEMIC_FEATURES:
            category = "Academic"
        elif feature in DEMOGRAPHIC_FEATURES:
            category = "Demographic"
        elif feature in INSTITUTIONAL_FEATURES:
            category = "Institutional/Course"
        elif feature in MACRO_FEATURES:
            category = "Macro-Economic"
        else:
            category = "Engagement"
        category_map[feature] = category
    return category_map


def pretty_name(feature: str) -> str:
    return DISPLAY_NAMES.get(feature, feature.replace("_", " ").strip().capitalize())


FRACTION_FEATURES = {
    "approval_rate_sem1",
    "approval_rate_sem2",
    "overall_approval_rate",
    "approval_rate_difference_sem2_minus_sem1",
    "evaluation_to_enrolled_ratio_sem1",
    "evaluation_to_enrolled_ratio_sem2",
    "overall_evaluation_to_enrolled_ratio",
}

GRADE_FEATURES = {
    "effective_avg_grade_sem1",
    "effective_avg_grade_sem2",
    "overall_average_grade",
    "Admission grade",
    "Previous qualification (grade)",
}

BINARY_LABELS = {
    "Debtor": ("No outstanding debt", "Outstanding debt"),
    "Tuition fees up to date": ("Tuition fees not up to date", "Tuition fees up to date"),
    "Scholarship holder": ("No scholarship support", "Scholarship holder"),
    "Displaced": ("Not displaced", "Displaced student"),
    "Educational special needs": ("No educational special needs", "Educational special needs"),
    "International": ("Domestic student", "International student"),
    "Daytime/evening attendance": ("Evening attendance", "Daytime attendance"),
    "Gender": ("Female", "Male"),
}

BINARY_OPTION_LABELS = {
    "Daytime/evening attendance": ("Evening", "Daytime"),
    "Gender": ("Female", "Male"),
    "Displaced": ("No", "Yes"),
    "Debtor": ("No", "Yes"),
    "Tuition fees up to date": ("No", "Yes"),
    "Scholarship holder": ("No", "Yes"),
}


def binary_options(field: str):
    """Return (label, value) choices for a binary form field."""
    no_label, yes_label = BINARY_OPTION_LABELS.get(field, ("No", "Yes"))
    return [(no_label, 0), (yes_label, 1)]


def field_options(field: str):
    """Return (display label, code) choices for a categorical form field."""
    if field == "Course":
        return [
            (label, code)
            for code, label in sorted(
                COURSE_LABELS.items(), key=lambda item: item[1]
            )
        ]
    if field in BINARY_FLAG_COLS:
        return binary_options(field)
    return None


COUNT_FEATURES = {
    "Age at enrollment",
    "Application order",
    "total_enrolled_units",
    "total_approved_units",
    "total_credited_units",
    "total_evaluations",
    "total_without_evaluations",
    "failed_units_sem1",
    "failed_units_sem2",
    "total_failed_units",
    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (credited)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (without evaluations)",
}


def format_value(feature: str, value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "missing"
    if feature in FRACTION_FEATURES:
        return f"{float(value):.0%}"
    if feature in {"Unemployment rate", "Inflation rate"}:
        return f"{float(value):.2f}%"
    if feature == "GDP":
        return f"{float(value):.2f}"
    if feature in COUNT_FEATURES:
        return str(int(round(float(value))))
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.2f}"
    return str(value)


def reason_text(feature: str, value, direction: str) -> str:
    """Turn a feature and its SHAP direction into a short, value-aware reason."""
    label = pretty_name(feature)
    increasing = direction == "increases risk"

    if feature in {"overall_approval_rate", "approval_rate_sem1", "approval_rate_sem2"}:
        rate = float(value)
        quality = "Low" if rate < 0.5 else ("Moderate" if rate < 0.8 else "High")
        return f"{quality} {label.lower()} ({format_value(feature, value)})"
    if feature in {"no_approved_sem1", "no_approved_sem2", "no_approved_both_semesters"}:
        return label if int(value) == 1 else f"Approved units present ({label.lower()})"
    if feature in {"low_approval_sem1", "low_approval_sem2", "low_approval_both_semesters"}:
        return label if int(value) == 1 else f"Approval rate is not low ({label.lower()})"
    if feature in {"failed_units_sem1", "failed_units_sem2", "total_failed_units"}:
        return f"{label} ({format_value(feature, value)})"
    if feature in {"effective_avg_grade_sem1", "effective_avg_grade_sem2", "overall_average_grade"}:
        grade = float(value)
        if grade <= 0:
            return f"No graded approved units ({label.lower()})"
        quality = "Low" if grade < 10 else ("Moderate" if grade < 14 else "High")
        return f"{quality} {label.lower()} ({format_value(feature, value)})"
    if feature == "grade_difference_sem2_minus_sem1":
        return f"Semester 2 grade change ({format_value(feature, value)})"
    if feature == "approved_difference_sem2_minus_sem1":
        return f"Semester 2 approved-unit change ({format_value(feature, value)})"
    if feature == "approval_rate_difference_sem2_minus_sem1":
        return f"Semester 2 approval-rate change ({format_value(feature, value)})"
    if feature == "failed_difference_sem2_minus_sem1":
        return f"Semester 2 failed-unit change ({format_value(feature, value)})"
    if feature == "Gender":
        return f"Gender: {BINARY_LABELS['Gender'][1 if int(value) == 1 else 0]}"
    if feature in BINARY_LABELS:
        return BINARY_LABELS[feature][1 if int(value) == 1 else 0]
    if feature in {"Admission grade", "Previous qualification (grade)"}:
        return f"{label} ({format_value(feature, value)})"
    if feature == "Age at enrollment":
        return f"Age at enrollment ({format_value(feature, value)})"
    if feature == "Course":
        code = int(float(value))
        return f"Course: {COURSE_LABELS.get(code, f'code {code}')}"
    if feature in CATEGORICAL_CODE_COLS:
        return f"{label} (code {int(float(value))})"
    if feature in {"Curricular units 1st sem (approved)", "Curricular units 2nd sem (approved)"}:
        count = int(value)
        semester = "1" if "1st" in feature else "2"
        return f"{count} approved units in semester {semester}"
    if feature in {"Curricular units 1st sem (enrolled)", "Curricular units 2nd sem (enrolled)"}:
        count = int(value)
        semester = "1" if "1st" in feature else "2"
        return f"{count} enrolled units in semester {semester}"

    suffix = " - increases risk" if increasing else " - reduces risk"
    return f"{label} = {format_value(feature, value)}{suffix}"

In [ ]:
from src.feature_engineering import (
    TARGET_COLS,
    CATEGORICAL_CODE_COLS,
    BINARY_FLAG_COLS,
    MACRO_COLS,
    engineer_features,
    get_feature_columns,
    build_category_map,
    pretty_name,
    binary_options,
)

RAW_FEATURES = [column for column in df.columns if column not in TARGET_COLS]
assert not set(RAW_FEATURES) & set(TARGET_COLS), "Target leakage detected."
assert all(
    column in df.columns
    for column in CATEGORICAL_CODE_COLS + BINARY_FLAG_COLS + MACRO_COLS
)

print("Raw features:", len(RAW_FEATURES))
print("Categorical codes:", len(CATEGORICAL_CODE_COLS))
print("Binary flags:", len(BINARY_FLAG_COLS))
print("Macro-economic:", len(MACRO_COLS))
print(
    "Other numeric:",
    len(RAW_FEATURES)
    - len(CATEGORICAL_CODE_COLS)
    - len(BINARY_FLAG_COLS)
    - len(MACRO_COLS),
)

df_fe = engineer_features(df)
FEATURE_COLUMNS = get_feature_columns(df_fe)
ENGINEERED_FEATURES = [
    column for column in FEATURE_COLUMNS if column not in RAW_FEATURES
]
CATEGORICAL_FEATURES = [
    column for column in CATEGORICAL_CODE_COLS if column in FEATURE_COLUMNS
]
NUMERIC_FEATURES = [
    column for column in FEATURE_COLUMNS if column not in CATEGORICAL_FEATURES
]
CATEGORY_MAP = build_category_map(FEATURE_COLUMNS)

assert not set(FEATURE_COLUMNS) & set(TARGET_COLS)
assert set(FEATURE_COLUMNS) == set(RAW_FEATURES) | set(ENGINEERED_FEATURES)
assert all(column in CATEGORY_MAP for column in FEATURE_COLUMNS)

print("\nTotal features:", len(FEATURE_COLUMNS))
print("Engineered features:", len(ENGINEERED_FEATURES))
print("\nEngineered feature names:")
print(ENGINEERED_FEATURES)
print("\nCategory counts:")
print(pd.Series(CATEGORY_MAP).value_counts())
display(df_fe[ENGINEERED_FEATURES].describe().T.round(2))

## Step 3 - Train / validation / test split

We use a stratified 70/15/15 split on `is_dropout`. The validation set is for
early stopping and threshold selection. The test set stays untouched until the
final evaluation.

In [ ]:
X = df_fe[FEATURE_COLUMNS].copy()
y = df_fe["is_dropout"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

assert len(set(X_train.index) & set(X_valid.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_valid.index) & set(X_test.index)) == 0

print(f"Train: {X_train.shape} | dropout rate: {y_train.mean():.3f}")
print(f"Valid: {X_valid.shape} | dropout rate: {y_valid.mean():.3f}")
print(f"Test:  {X_test.shape} | dropout rate: {y_test.mean():.3f}")

## Step 4 - Preprocessing module

Categorical codes use frequency encoding learned from the training set. Missing
numeric values use the training median. No scaling is needed for XGBoost. The
fitted object is a plain dictionary, which makes it easy to save and load in the
notebook, the prediction module, and Streamlit.

In [ ]:
%%writefile src/preprocessing.py
"""Leakage-safe preprocessing: frequency encoding and median imputation."""

from __future__ import annotations

import numpy as np
import pandas as pd


def fit_preprocessor(
    X_train: pd.DataFrame,
    categorical_features,
    numeric_features,
    feature_columns,
) -> dict:
    """Learn category frequencies and numeric medians from the training set."""
    frequency_maps = {}
    for column in categorical_features:
        values = X_train[column].astype("string").fillna("missing")
        frequencies = values.value_counts(normalize=True)
        frequency_maps[column] = {
            str(key): float(value) for key, value in frequencies.items()
        }

    medians = {}
    for column in numeric_features:
        values = pd.to_numeric(X_train[column], errors="coerce")
        median = values.median()
        medians[column] = float(median) if pd.notna(median) else 0.0

    return {
        "categorical_features": list(categorical_features),
        "numeric_features": list(numeric_features),
        "frequency_maps": frequency_maps,
        "medians": medians,
        "feature_columns": list(feature_columns),
    }


def transform_features(data: pd.DataFrame, preprocessor: dict) -> pd.DataFrame:
    """Apply a fitted preprocessor to any split or to one new student."""
    output = pd.DataFrame(index=data.index)

    for column in preprocessor["categorical_features"]:
        values = data[column].astype("string").fillna("missing")
        mapping = preprocessor["frequency_maps"][column]
        output[column] = values.map(mapping).astype(float).fillna(0.0)

    for column in preprocessor["numeric_features"]:
        values = pd.to_numeric(data[column], errors="coerce")
        output[column] = values.fillna(preprocessor["medians"][column])

    output = output.reindex(columns=preprocessor["feature_columns"])
    output = output.replace([np.inf, -np.inf], np.nan)
    return output.fillna(0.0).astype(float)

In [ ]:
from src.preprocessing import fit_preprocessor, transform_features

preprocessor = fit_preprocessor(
    X_train,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    FEATURE_COLUMNS,
)

X_train_p = transform_features(X_train, preprocessor)
X_valid_p = transform_features(X_valid, preprocessor)
X_test_p = transform_features(X_test, preprocessor)

assert not X_train_p.isna().any().any()
assert not X_valid_p.isna().any().any()
assert not X_test_p.isna().any().any()
assert not np.isinf(X_train_p.to_numpy()).any()

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

print("Encoded shapes:", X_train_p.shape, X_valid_p.shape, X_test_p.shape)
print("\nTop course frequencies (learned from train only):")
print(
    pd.Series(preprocessor["frequency_maps"]["Course"])
      .sort_values(ascending=False)
      .head()
)
print("\nMissing values after preprocessing:", int(X_train_p.isna().sum().sum()))

## Step 5 - Train XGBoost

The binary target is not perfectly balanced, so `scale_pos_weight` is calculated
from the training split. Early stopping uses the validation AUC. The test set is
not involved.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(
    X_train_p,
    y_train,
    eval_set=[(X_valid_p, y_valid)],
    verbose=False,
)

valid_prob = model.predict_proba(X_valid_p)[:, 1]
test_prob = model.predict_proba(X_test_p)[:, 1]

print("scale_pos_weight:", round(scale_pos_weight, 3))
print("Best iteration:", model.best_iteration)
print("Best validation AUC:", round(float(model.best_score), 4))

## Step 6 - Choose a decision threshold on validation data

Youden's J (`TPR - FPR`) picks the threshold that balances catching at-risk
students against false alarms. The threshold is saved with the model.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_valid, valid_prob)
youden_j = tpr - fpr
best_index = int(np.argmax(youden_j))
best_threshold = float(thresholds[best_index])

if not np.isfinite(best_threshold) or best_threshold <= 0 or best_threshold >= 1:
    best_threshold = 0.5


def risk_level(probability):
    for level, (lower, upper) in RISK_BANDS.items():
        if lower <= probability < upper:
            return level
    return "Critical"


print(f"Best threshold (Youden's J): {best_threshold:.4f}")
print(f"Validation Youden's J: {youden_j[best_index]:.4f}")
print("Validation confusion matrix at tuned threshold:")
print(confusion_matrix(y_valid, (valid_prob >= best_threshold).astype(int)))

## Step 7 - Evaluate on validation and test

Recall for the dropout class is the key metric for early warning, but precision
is reported as well. Metrics at the tuned threshold and at the default 0.50
threshold are both saved.

In [ ]:
def to_python(obj):
    if isinstance(obj, dict):
        return {str(key): to_python(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_python(value) for value in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def evaluate_split(name, y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    result = {
        "split": name,
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "pr_auc": float(average_precision_score(y_true, probabilities)),
        "confusion_matrix": confusion_matrix(y_true, predictions).tolist(),
        "classification_report": classification_report(
            y_true,
            predictions,
            target_names=["Not dropout", "Dropout"],
            output_dict=True,
            zero_division=0,
        ),
    }
    print(f"\n{name.upper()} (threshold={threshold:.4f})")
    print(f"  Accuracy : {result['accuracy']:.4f}")
    print(f"  Precision: {result['precision']:.4f}")
    print(f"  Recall   : {result['recall']:.4f}")
    print(f"  F1       : {result['f1']:.4f}")
    print(f"  ROC AUC  : {result['roc_auc']:.4f}")
    print(f"  PR AUC   : {result['pr_auc']:.4f}")
    print("  Confusion matrix:")
    print(np.array(result["confusion_matrix"]))
    return result


validation_metrics = evaluate_split(
    "Validation", y_valid, valid_prob, best_threshold
)
test_metrics = evaluate_split("Test", y_test, test_prob, best_threshold)
validation_at_05 = evaluate_split(
    "Validation at 0.50", y_valid, valid_prob, 0.50
)
test_at_05 = evaluate_split("Test at 0.50", y_test, test_prob, 0.50)

evaluation_report = {
    "dataset": {
        "path": str(DATASET_PATH),
        "shape": list(df.shape),
        "target_distribution": df["Target"].value_counts().to_dict(),
    },
    "features": {
        "count": len(FEATURE_COLUMNS),
        "categorical_encoded": len(CATEGORICAL_FEATURES),
        "numeric": len(NUMERIC_FEATURES),
        "engineered": len(ENGINEERED_FEATURES),
    },
    "split": {
        "train": list(X_train.shape),
        "valid": list(X_valid.shape),
        "test": list(X_test.shape),
    },
    "threshold_selection": {
        "method": "Youden's J statistic on validation set",
        "best_threshold": best_threshold,
        "youden_j": float(youden_j[best_index]),
    },
    "validation": validation_metrics,
    "test": test_metrics,
    "validation_at_0_5": validation_at_05,
    "test_at_0_5": test_at_05,
    "risk_bands": RISK_BANDS,
}

with open(OUTPUTS_DIR / "evaluation_report.json", "w", encoding="utf-8") as file:
    json.dump(to_python(evaluation_report), file, indent=2)

print("\nSaved:", OUTPUTS_DIR / "evaluation_report.json")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, title, actual, probabilities in zip(
    axes,
    ["Validation", "Test"],
    [y_valid, y_test],
    [valid_prob, test_prob],
):
    matrix = confusion_matrix(actual, (probabilities >= best_threshold).astype(int))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axis,
        xticklabels=["Not dropout", "Dropout"],
        yticklabels=["Not dropout", "Dropout"],
    )
    axis.set_title(f"{title} confusion matrix")
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## Step 8 - XGBoost feature importance

This is the model's own split-based importance. It is useful as a quick ranking,
while SHAP gives the directional and individual explanations.

In [ ]:
importance_df = (
    pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "importance": model.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
importance_df.to_csv(OUTPUTS_DIR / "feature_importance.csv", index=False)

print(importance_df.head(15).to_string(index=False))

top_importance = importance_df.head(20).sort_values("importance")
plt.figure(figsize=(10, 7))
plt.barh(top_importance["feature"], top_importance["importance"], color="#5c6bc0")
plt.xlabel("XGBoost importance")
plt.title("Top 20 features")
plt.tight_layout()
plt.show()

## Step 9 - Global SHAP and category contribution

SHAP values are computed on the test sample. The global ranking uses mean
absolute SHAP, and the category view groups those values into Academic,
Engagement, Financial, Demographic, Institutional/Course, and Macro-Economic.

In [ ]:
EXPLAINER = shap.TreeExplainer(model)
shap_sample = (
    X_test_p
    if len(X_test_p) <= 1000
    else X_test_p.sample(1000, random_state=RANDOM_STATE)
)
shap_values = EXPLAINER.shap_values(shap_sample)

if isinstance(shap_values, list):
    shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
shap_values = np.asarray(shap_values)
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

mean_abs_shap = np.abs(shap_values).mean(axis=0)
global_importance = pd.DataFrame(
    {"feature": FEATURE_COLUMNS, "mean_abs_shap": mean_abs_shap}
)
global_importance["percent_contribution"] = (
    100
    * global_importance["mean_abs_shap"]
    / global_importance["mean_abs_shap"].sum()
)
global_importance = (
    global_importance
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
global_importance.to_csv(
    OUTPUTS_DIR / "shap_global_importance.csv", index=False
)

global_importance["category"] = global_importance["feature"].map(CATEGORY_MAP)
category_contribution = (
    global_importance.groupby("category", as_index=False)["mean_abs_shap"]
    .sum()
)
category_contribution["percent_contribution"] = (
    100
    * category_contribution["mean_abs_shap"]
    / category_contribution["mean_abs_shap"].sum()
)
category_contribution = (
    category_contribution
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
category_contribution.to_csv(
    OUTPUTS_DIR / "category_contribution.csv", index=False
)

print(global_importance.head(15).to_string(index=False))
print("\nCategory contribution:")
print(
    category_contribution[["category", "percent_contribution"]]
    .to_string(index=False)
)

shap.summary_plot(
    shap_values,
    shap_sample,
    feature_names=FEATURE_COLUMNS,
    max_display=20,
    show=False,
)
plt.title("SHAP global importance (test sample)")
plt.tight_layout()
plt.show()

## Step 10 - Save the model artifacts

The model, feature list, threshold, category map, metadata, and default input
values are saved. Defaults are training medians for numeric fields and training
modes for encoded categorical and binary fields. The app uses them when a field
is missing.

In [ ]:
joblib.dump(model, MODELS_DIR / "dropout_model.joblib")

with open(MODELS_DIR / "feature_columns.json", "w", encoding="utf-8") as file:
    json.dump(FEATURE_COLUMNS, file, indent=2)

with open(MODELS_DIR / "threshold.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "best_threshold": best_threshold,
            "method": "Youden's J statistic on validation set",
            "youden_j": float(youden_j[best_index]),
            "risk_bands": RISK_BANDS,
        },
        file,
        indent=2,
    )

with open(MODELS_DIR / "category_map.json", "w", encoding="utf-8") as file:
    json.dump(CATEGORY_MAP, file, indent=2)

train_raw = df.loc[X_train.index, RAW_FEATURES]
default_input_values = {}
for column in RAW_FEATURES:
    if column in CATEGORICAL_CODE_COLS or column in BINARY_FLAG_COLS:
        value = train_raw[column].mode(dropna=True).iloc[0]
        default_input_values[column] = int(value)
    else:
        value = pd.to_numeric(train_raw[column], errors="coerce").median()
        default_input_values[column] = float(value) if pd.notna(value) else 0.0

with open(
    MODELS_DIR / "default_input_values.json", "w", encoding="utf-8"
) as file:
    json.dump(default_input_values, file, indent=2)

metadata = {
    "model_type": "xgboost",
    "target": "is_dropout",
    "random_state": RANDOM_STATE,
    "feature_count": len(FEATURE_COLUMNS),
    "train_shape": list(X_train.shape),
    "valid_shape": list(X_valid.shape),
    "test_shape": list(X_test.shape),
    "best_threshold": best_threshold,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_shape": list(df.shape),
    "xgboost_version": xgb.__version__,
    "shap_version": shap.__version__,
    "test_metrics": {
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "roc_auc": test_metrics["roc_auc"],
        "pr_auc": test_metrics["pr_auc"],
    },
}
with open(MODELS_DIR / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved model artifacts:")
for path in sorted(MODELS_DIR.glob("*")):
    print(" -", path.name)

## Step 11 - Prediction and explanation module

This module loads the artifacts, fills missing fields with training defaults,
engineers features, predicts, computes individual SHAP values, and builds the
recommendation list. The same module is used by the Kaggle assistant and by the
local Streamlit app.

In [ ]:
%%writefile src/predict.py
"""Load saved artifacts, predict dropout risk, and explain the result."""

from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap

from .feature_engineering import engineer_features, pretty_name, reason_text
from .preprocessing import transform_features

PROJECT_ROOT = Path(__file__).resolve().parents[1]
DEFAULT_MODELS_DIR = PROJECT_ROOT / "models"

DISCLAIMER = (
    "This prediction is a decision-support tool and should not be used as the "
    "final decision. Human review is required."
)

RECOMMENDATIONS = {
    "Academic": [
        "Arrange tutoring or extra academic support.",
        "Review previous qualification readiness.",
        "Contact course teachers for academic feedback.",
        "Create a grade improvement plan.",
    ],
    "Engagement": [
        "Contact the student for a supportive check-in.",
        "Create a study recovery plan.",
        "Monitor assignment and exam participation.",
        "Assign a peer buddy or mentor.",
        "Schedule a counselor meeting.",
    ],
    "Financial": [
        "Check scholarship eligibility.",
        "Contact the financial aid office.",
        "Offer payment plan information.",
        "Verify tuition fee status.",
    ],
    "Demographic": [
        "Offer student support services.",
        "Connect the student with counseling.",
        "Provide peer support or mentorship.",
        "Check whether the student needs relocation or special assistance.",
    ],
    "Institutional/Course": [
        "Contact the course coordinator.",
        "Review course difficulty or workload.",
        "Check whether the student is in the correct course.",
        "Offer academic advising.",
    ],
    "Macro-Economic": [
        "Provide institutional support resources.",
        "Monitor the student more closely.",
        "Offer career and financial counseling if available.",
    ],
}


def _load_json(path: Path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def _is_missing(value) -> bool:
    if value is None:
        return True
    if isinstance(value, str) and not value.strip():
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


class DropoutPredictor:
    """Reusable predictor for one student or a Streamlit form."""

    def __init__(self, models_dir=None):
        self.models_dir = Path(models_dir) if models_dir else DEFAULT_MODELS_DIR
        self.model = joblib.load(self.models_dir / "dropout_model.joblib")
        self.preprocessor = joblib.load(self.models_dir / "preprocessor.joblib")
        self.feature_columns = _load_json(self.models_dir / "feature_columns.json")
        threshold_info = _load_json(self.models_dir / "threshold.json")
        self.threshold = float(threshold_info["best_threshold"])
        self.risk_bands = threshold_info.get(
            "risk_bands",
            {
                "Low": [0.00, 0.35],
                "Medium": [0.35, 0.65],
                "High": [0.65, 0.85],
                "Critical": [0.85, 1.01],
            },
        )
        self.category_map = _load_json(self.models_dir / "category_map.json")
        self.default_input_values = _load_json(
            self.models_dir / "default_input_values.json"
        )
        self.raw_features = list(self.default_input_values.keys())
        self.explainer = shap.TreeExplainer(self.model)

    def risk_level(self, probability: float) -> str:
        for level, band in self.risk_bands.items():
            lower, upper = band
            if lower <= probability < upper:
                return level
        return "Critical"

    def _shap_values(self, X_row: pd.DataFrame) -> np.ndarray:
        values = self.explainer.shap_values(X_row)
        if isinstance(values, list):
            values = values[1] if len(values) > 1 else values[0]
        values = np.asarray(values)
        if values.ndim == 3:
            values = values[:, :, 1]
        return values.reshape(-1)

    def explain_row(
        self,
        raw_row,
        feature_row,
        X_row,
        top_n=5,
        risk_level=None,
        allowed_features=None,
    ):
        shap_values = self._shap_values(X_row)
        if shap_values.shape[0] != len(self.feature_columns):
            raise ValueError("SHAP output does not match the saved feature columns.")

        allowed = set(allowed_features) if allowed_features is not None else None
        records = []

        for index, feature in enumerate(self.feature_columns):
            if allowed is not None and feature not in allowed:
                continue
            if feature in raw_row.columns:
                value = raw_row.iloc[0][feature]
            else:
                value = feature_row.iloc[0].get(feature, np.nan)
            if hasattr(value, "item"):
                value = value.item()

            shap_value = float(shap_values[index])
            direction = "increases risk" if shap_value > 0 else "decreases risk"
            records.append(
                {
                    "feature": feature,
                    "feature_label": pretty_name(feature),
                    "category": self.category_map.get(feature, "Engagement"),
                    "value": value,
                    "encoded_value": round(float(X_row.iloc[0][feature]), 6),
                    "shap_value": round(shap_value, 6),
                    "direction": direction,
                    "reason": reason_text(feature, value, direction),
                }
            )

        if not records:
            return [], []

        total = sum(abs(row["shap_value"]) for row in records) or 1.0
        for row in records:
            row["percent_contribution"] = round(
                100 * abs(row["shap_value"]) / total, 2
            )

        positive = sorted(
            [row for row in records if row["shap_value"] > 0],
            key=lambda row: row["shap_value"],
            reverse=True,
        )
        negative = sorted(
            [row for row in records if row["shap_value"] <= 0],
            key=lambda row: abs(row["shap_value"]),
            reverse=True,
        )
        if risk_level == "Low":
            # For a low-risk student, show the strongest factors overall,
            # including protective factors, instead of only risk-increasing ones.
            top_reasons = sorted(
                records, key=lambda row: abs(row["shap_value"]), reverse=True
            )[:top_n]
        else:
            top_reasons = (positive + negative)[:top_n]

        category_totals = {}
        for row in records:
            category = row["category"]
            category_totals[category] = (
                category_totals.get(category, 0.0) + abs(row["shap_value"])
            )
        category_contributions = [
            {
                "category": category,
                "percent_contribution": round(100 * value / total, 2),
            }
            for category, value in sorted(
                category_totals.items(), key=lambda item: item[1], reverse=True
            )
        ]
        return top_reasons, category_contributions

    def recommend(self, category_contributions, top_reasons, risk_level):
        if risk_level == "Low":
            return [
                "No immediate intervention is indicated.",
                "Continue routine monitoring and encourage current academic habits.",
                "Share available academic, financial, and wellbeing support resources.",
            ]

        top_category = (
            category_contributions[0]["category"]
            if category_contributions
            else "Engagement"
        )
        recommendations = list(
            RECOMMENDATIONS.get(top_category, RECOMMENDATIONS["Engagement"])
        )
        features = {row["feature"] for row in top_reasons}

        if features & {"Debtor", "Tuition fees up to date"}:
            recommendations.append(
                "Review financial aid, scholarship, and payment-plan options."
            )
        if features & {
            "overall_approval_rate",
            "total_failed_units",
            "failed_units_sem1",
            "failed_units_sem2",
        }:
            recommendations.append(
                "Create a semester recovery plan with the academic advisor."
            )
        if risk_level in {"High", "Critical"}:
            recommendations.append(
                "Schedule a supportive check-in with the student soon."
            )

        unique = []
        for item in recommendations:
            if item not in unique:
                unique.append(item)
        return unique[:6]

    def predict(self, input_data: dict, explanation_features=None) -> dict:
        """Predict one student and return a JSON-compatible explanation.

        Pass ``explanation_features`` to limit SHAP reasons to the fields that
        were actually collected by a simplified form.
        """
        input_data = dict(input_data or {})
        unknown_fields = [
            field for field in input_data if field not in self.raw_features
        ]
        missing_fields = []
        values = {}

        for feature in self.raw_features:
            value = input_data.get(feature)
            if _is_missing(value):
                value = self.default_input_values[feature]
                missing_fields.append(feature)
            values[feature] = value

        raw_row = pd.DataFrame([values], columns=self.raw_features)
        raw_row = raw_row.replace(
            {"Yes": 1, "No": 0, "yes": 1, "no": 0, "TRUE": 1, "FALSE": 0}
        )
        for column in self.raw_features:
            raw_row[column] = pd.to_numeric(raw_row[column], errors="coerce")
            if raw_row[column].isna().any():
                raw_row[column] = raw_row[column].fillna(
                    self.default_input_values[column]
                )

        feature_row = engineer_features(raw_row)
        X_row = transform_features(feature_row, self.preprocessor)
        X_row = X_row.reindex(columns=self.feature_columns)

        probability = float(self.model.predict_proba(X_row)[0, 1])
        risk = self.risk_level(probability)
        top_reasons, category_contributions = self.explain_row(
            raw_row,
            feature_row,
            X_row,
            risk_level=risk,
            allowed_features=explanation_features,
        )
        recommendations = self.recommend(
            category_contributions, top_reasons, risk
        )

        return {
            "dropout_probability": round(probability, 4),
            "dropout_probability_percent": round(probability * 100, 1),
            "risk_level": risk,
            "predicted_dropout": int(probability >= self.threshold),
            "threshold_used": round(self.threshold, 4),
            "top_reasons": top_reasons,
            "category_contributions": category_contributions,
            "recommendations": recommendations,
            "missing_fields_filled": missing_fields,
            "ignored_fields": unknown_fields,
            "disclaimer": DISCLAIMER,
        }


_DEFAULT_PREDICTOR = None


def get_predictor() -> DropoutPredictor:
    global _DEFAULT_PREDICTOR
    if _DEFAULT_PREDICTOR is None:
        _DEFAULT_PREDICTOR = DropoutPredictor()
    return _DEFAULT_PREDICTOR


def predict_dropout(input_data: dict, explanation_features=None) -> dict:
    """Convenience function used by the chatbot and notebooks."""
    return get_predictor().predict(
        input_data, explanation_features=explanation_features
    )

In [ ]:
from src.feature_engineering import (
    APP_EXPLANATION_FEATURES,
    APP_SECTIONS,
    INT_INPUT_FIELDS,
    field_options,
)
from src.predict import DropoutPredictor

predictor = DropoutPredictor(models_dir=MODELS_DIR)

default_result = predictor.predict(
    default_input_values, explanation_features=APP_EXPLANATION_FEATURES
)
print("Default sample student")
print(json.dumps(default_result, indent=2, default=str)[:2500])

high_risk_student = dict(default_input_values)
high_risk_student.update(
    {
        "Curricular units 1st sem (enrolled)": 6,
        "Curricular units 1st sem (approved)": 0,
        "Curricular units 1st sem (grade)": 0.0,
        "Curricular units 2nd sem (enrolled)": 6,
        "Curricular units 2nd sem (approved)": 0,
        "Curricular units 2nd sem (grade)": 0.0,
        "Debtor": 1,
        "Tuition fees up to date": 0,
        "Scholarship holder": 0,
        "Admission grade": 95.0,
    }
)

low_risk_student = dict(default_input_values)
low_risk_student.update(
    {
        "Curricular units 1st sem (enrolled)": 6,
        "Curricular units 1st sem (approved)": 6,
        "Curricular units 1st sem (grade)": 16.0,
        "Curricular units 2nd sem (enrolled)": 6,
        "Curricular units 2nd sem (approved)": 6,
        "Curricular units 2nd sem (grade)": 16.0,
        "Debtor": 0,
        "Tuition fees up to date": 1,
        "Scholarship holder": 1,
        "Admission grade": 150.0,
    }
)

high_result = predictor.predict(
    high_risk_student, explanation_features=APP_EXPLANATION_FEATURES
)
low_result = predictor.predict(
    low_risk_student, explanation_features=APP_EXPLANATION_FEATURES
)

for name, result in [
    ("High-risk example", high_result),
    ("Low-risk example", low_result),
]:
    print(
        f"\n{name}: {result['dropout_probability_percent']}% "
        f"({result['risk_level']} risk)"
    )
    print("Top reasons:")
    for item in result["top_reasons"]:
        print(
            f"  - {item['reason']} | {item['category']} | "
            f"{item['percent_contribution']:.1f}%"
        )

assert high_result["dropout_probability"] > low_result["dropout_probability"]

with open(
    OUTPUTS_DIR / "example_prediction.json", "w", encoding="utf-8"
) as file:
    json.dump(high_result, file, indent=2, default=str)

print("\nSaved:", OUTPUTS_DIR / "example_prediction.json")
print("\nPrediction module checks passed.")

## Step 12 - Kaggle chatbot (ipywidgets)

Fill the 14 core fields, click **Predict Risk**, and the assistant shows the
probability, risk level, key factors, category chart, and recommended actions.
Course names, female/male, and no/yes choices are shown as text instead of
numeric codes. The chat box accepts `predict`, `use default student`,
`explain risk`, and `show recommendations`.

Only the fields in the form are shown as key factors. Other background fields
(such as marital status, nationality, family occupation, application details,
and macro indicators) are filled with typical training values. Kaggle does not
render Streamlit inline, so this ipywidgets version is the in-notebook chatbot;
the Streamlit app is exported for the later local shift.

In [ ]:
FORM_SECTIONS = APP_SECTIONS

RISK_COLORS = {
    "Low": "#2e7d32",
    "Medium": "#f9a825",
    "High": "#ef6c00",
    "Critical": "#c62828",
}

form_widgets = {}
section_boxes = []

for section, fields in FORM_SECTIONS.items():
    rows = []
    for field in fields:
        default = default_input_values[field]
        options = field_options(field)
        if options:
            widget = widgets.Dropdown(options=options, value=int(default))
        elif field in INT_INPUT_FIELDS:
            widget = widgets.IntText(value=int(default))
        else:
            widget = widgets.FloatText(value=float(default))
        widget.layout = widgets.Layout(width="310px")
        form_widgets[field] = widget
        label = widgets.HTML(
            "<div style='width:285px;font-size:12px;line-height:1.2'>"
            f"{pretty_name(field)}</div>"
        )
        rows.append(
            widgets.HBox(
                [label, widget],
                layout=widgets.Layout(align_items="center", margin="2px 0"),
            )
        )
    section_boxes.append(widgets.VBox(rows))

accordion = widgets.Accordion(
    children=section_boxes, titles=list(FORM_SECTIONS.keys())
)
accordion.selected_index = 0

predict_button = widgets.Button(description="Predict Risk", button_style="success")
default_button = widgets.Button(description="Use Default Student")
high_button = widgets.Button(description="High-Risk Example")
low_button = widgets.Button(description="Low-Risk Example")
output = widgets.Output()
last_result = {"value": None}


def read_form():
    return {field: widget.value for field, widget in form_widgets.items()}


def read_full_form():
    return {**default_input_values, **read_form()}


def form_errors(values):
    errors = []
    for semester, label in (("1st", "1"), ("2nd", "2")):
        enrolled = values[f"Curricular units {semester} sem (enrolled)"]
        approved = values[f"Curricular units {semester} sem (approved)"]
        if approved > enrolled:
            errors.append(
                f"Semester {label}: approved units cannot exceed enrolled units."
            )
    return errors


def write_form(values):
    for field, widget in form_widgets.items():
        widget.value = values.get(field, default_input_values[field])


def display_result(result):
    last_result["value"] = result
    output.clear_output()
    with output:
        color = RISK_COLORS.get(result["risk_level"], "#555555")
        display(
            HTML(
                f"""
                <div style="border:1px solid #ddd;border-radius:12px;padding:16px;
                            margin:8px 0;background:#fafafa">
                    <div style="font-size:13px;color:#666">Estimated dropout probability</div>
                    <div style="font-size:38px;font-weight:700">{result['dropout_probability_percent']}%</div>
                    <div style="font-size:18px;color:{color};font-weight:600">
                        Risk level: {result['risk_level']}
                    </div>
                    <div style="font-size:12px;color:#777">
                        Decision threshold: {result['threshold_used']:.2f}
                    </div>
                </div>
                """
            )
        )
        reason_rows = "".join(
            f"<tr><td>{index}</td><td>{item['reason']}</td>"
            f"<td>{item['category']}</td>"
            f"<td>{item['percent_contribution']:.1f}%</td>"
            f"<td>{item['direction']}</td></tr>"
            for index, item in enumerate(result["top_reasons"], 1)
        )
        display(
            HTML(
                f"""
                <h3 style="margin:10px 0 6px">Key factors</h3>
                <table style="width:100%;border-collapse:collapse;font-size:13px">
                    <tr style="background:#eef1f7">
                        <th align="left">#</th><th align="left">Reason</th>
                        <th align="left">Category</th>
                        <th align="left">Contribution</th>
                        <th align="left">Direction</th>
                    </tr>
                    {reason_rows}
                </table>
                """
            )
        )
        category_df = pd.DataFrame(result["category_contributions"])
        if not category_df.empty:
            axis = (
                category_df.set_index("category")["percent_contribution"]
                .plot(kind="barh", figsize=(7, 3), color="#5c6bc0")
            )
            axis.invert_yaxis()
            axis.set_xlabel("Share of explanation (%)")
            axis.set_ylabel("")
            plt.tight_layout()
            plt.show()
            plt.close()
        display(
            HTML(
                "<h3 style='margin:10px 0 6px'>Recommended actions</h3><ul>"
                + "".join(
                    f"<li>{item}</li>" for item in result["recommendations"]
                )
                + "</ul>"
            )
        )
        display(
            HTML(
                "<p style='color:#777;font-size:12px'>Only the fields in the "
                "form are shown as key factors. Other background fields use "
                "typical training values.</p>"
            )
        )
        display(
            HTML(
                f"<p style='color:#777;font-size:12px'>{result['disclaimer']}</p>"
            )
        )


def on_predict(_):
    values = read_form()
    errors = form_errors(values)
    if errors:
        output.clear_output()
        with output:
            for error in errors:
                display(HTML(f"<p style='color:#c62828'>{error}</p>"))
        return
    display_result(
        predictor.predict(
            read_full_form(), explanation_features=APP_EXPLANATION_FEATURES
        )
    )


def on_default(_):
    write_form(default_input_values)
    display_result(
        predictor.predict(
            default_input_values,
            explanation_features=APP_EXPLANATION_FEATURES,
        )
    )


def on_high(_):
    write_form(high_risk_student)
    display_result(
        predictor.predict(
            high_risk_student,
            explanation_features=APP_EXPLANATION_FEATURES,
        )
    )


def on_low(_):
    write_form(low_risk_student)
    display_result(
        predictor.predict(
            low_risk_student,
            explanation_features=APP_EXPLANATION_FEATURES,
        )
    )


for button, callback in [
    (predict_button, on_predict),
    (default_button, on_default),
    (high_button, on_high),
    (low_button, on_low),
]:
    button.on_click(callback)

chat_input = widgets.Text(
    placeholder="Try: predict | use default student | explain risk | show recommendations",
    layout=widgets.Layout(width="560px"),
)
chat_send = widgets.Button(description="Send")
chat_output = widgets.Output()


def on_chat(_):
    text = chat_input.value.strip()
    if not text:
        return
    command = text.lower()
    chat_input.value = ""
    reply = ""

    if command in {"predict", "predict risk"}:
        values = read_form()
        errors = form_errors(values)
        if errors:
            reply = "Please fix the form first: " + " ".join(errors)
        else:
            result = predictor.predict(
                read_full_form(), explanation_features=APP_EXPLANATION_FEATURES
            )
            display_result(result)
            reply = (
                f"Estimated dropout probability: "
                f"{result['dropout_probability_percent']}% "
                f"({result['risk_level']} risk)."
            )
    elif command in {"use default student", "default"}:
        write_form(default_input_values)
        result = predictor.predict(
            default_input_values,
            explanation_features=APP_EXPLANATION_FEATURES,
        )
        display_result(result)
        reply = (
            "Loaded the default student. "
            f"Estimated probability: {result['dropout_probability_percent']}% "
            f"({result['risk_level']} risk)."
        )
    elif command in {"explain risk", "explain"}:
        result = last_result["value"]
        if result:
            reply = "<br>".join(
                f"{index}. {item['reason']} "
                f"({item['category']}, {item['percent_contribution']:.1f}%)"
                for index, item in enumerate(result["top_reasons"][:5], 1)
            )
        else:
            reply = "Please run a prediction first."
    elif command in {"show recommendations", "recommendations"}:
        result = last_result["value"]
        reply = (
            "<br>".join(f"- {item}" for item in result["recommendations"])
            if result
            else "Please run a prediction first."
        )
    else:
        reply = (
            "Try: predict, use default student, explain risk, "
            "show recommendations."
        )

    with chat_output:
        display(HTML(f"<b>You:</b> {text}"))
        display(HTML(f"<b>Assistant:</b> {reply}"))


chat_send.on_click(on_chat)
chat_input.on_submit(on_chat)

display(
    HTML(
        "<h3>Student Dropout Risk Assistant</h3>"
        "<p>Enter student information, then click <b>Predict Risk</b>. "
        "The assistant estimates risk and explains the main factors.</p>"
    )
)
display(widgets.HBox([predict_button, default_button, high_button, low_button]))
display(accordion)
display(output)
display(HTML("<h4>Ask the assistant</h4>"))
display(widgets.HBox([chat_input, chat_send]))
display(chat_output)

## Step 13 - Export the Streamlit app and download the project

The app loads the saved artifacts and never retrains. It includes the same
prediction, SHAP explanation, recommendation, and chat-command behavior in a
local Streamlit interface. This cell writes the app, README, requirements, and a
ZIP of the whole project.

In [ ]:
%%writefile app/chatbot_app.py
"""Streamlit chatbot for student dropout risk (local version)."""

from __future__ import annotations

import functools
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.feature_engineering import (
    APP_EXPLANATION_FEATURES,
    APP_INPUT_FIELDS,
    APP_SECTIONS,
    INT_INPUT_FIELDS,
    field_options,
    pretty_name,
)
from src.predict import get_predictor

st.set_page_config(
    page_title="Student Dropout Risk Assistant",
    layout="wide",
)


@st.cache_resource
def load_predictor():
    return get_predictor()


try:
    predictor = load_predictor()
except Exception as exc:
    st.error(
        "Could not load the model artifacts. Run the training notebook first "
        "so the models/ folder is populated."
    )
    st.exception(exc)
    st.stop()

DEFAULTS = predictor.default_input_values
ALL_FEATURES = predictor.raw_features
INPUT_FIELDS = list(APP_INPUT_FIELDS)
HIDDEN_FIELDS = [field for field in ALL_FEATURES if field not in INPUT_FIELDS]

FIELD_HELP = {
    "Course": "Select the course the student is enrolled in.",
    "Age at enrollment": "Age when the student started the course.",
    "Gender": "Recorded in the dataset as female or male.",
    "Displaced": "The student moved away from home to study.",
    "Admission grade": "Admission score on the 0-200 scale.",
    "Curricular units 1st sem (enrolled)": "Units enrolled in semester 1.",
    "Curricular units 1st sem (approved)": "Units approved in semester 1.",
    "Curricular units 1st sem (grade)": "Average grade in semester 1 (0-20).",
    "Curricular units 2nd sem (enrolled)": "Units enrolled in semester 2.",
    "Curricular units 2nd sem (approved)": "Units approved in semester 2.",
    "Curricular units 2nd sem (grade)": "Average grade in semester 2 (0-20).",
    "Tuition fees up to date": "Whether tuition fees are currently paid.",
    "Debtor": "Whether the student has outstanding debt.",
    "Scholarship holder": "Whether the student receives a scholarship.",
}

FIELD_RANGES = {
    "Age at enrollment": (16, 80, 1),
    "Admission grade": (0.0, 200.0, 0.1),
    "Curricular units 1st sem (enrolled)": (0, 30, 1),
    "Curricular units 1st sem (approved)": (0, 30, 1),
    "Curricular units 1st sem (grade)": (0.0, 20.0, 0.1),
    "Curricular units 2nd sem (enrolled)": (0, 30, 1),
    "Curricular units 2nd sem (approved)": (0, 30, 1),
    "Curricular units 2nd sem (grade)": (0.0, 20.0, 0.1),
}

HIGH_RISK_UPDATES = {
    "Course": 9500,
    "Age at enrollment": 21,
    "Gender": 0,
    "Displaced": 0,
    "Admission grade": 95.0,
    "Curricular units 1st sem (enrolled)": 6,
    "Curricular units 1st sem (approved)": 0,
    "Curricular units 1st sem (grade)": 0.0,
    "Curricular units 2nd sem (enrolled)": 6,
    "Curricular units 2nd sem (approved)": 0,
    "Curricular units 2nd sem (grade)": 0.0,
    "Tuition fees up to date": 0,
    "Debtor": 1,
    "Scholarship holder": 0,
}

LOW_RISK_UPDATES = {
    "Course": 9500,
    "Age at enrollment": 19,
    "Gender": 0,
    "Displaced": 0,
    "Admission grade": 150.0,
    "Curricular units 1st sem (enrolled)": 6,
    "Curricular units 1st sem (approved)": 6,
    "Curricular units 1st sem (grade)": 16.0,
    "Curricular units 2nd sem (enrolled)": 6,
    "Curricular units 2nd sem (approved)": 6,
    "Curricular units 2nd sem (grade)": 16.0,
    "Tuition fees up to date": 1,
    "Debtor": 0,
    "Scholarship holder": 1,
}


def field_key(field: str) -> str:
    return f"input_{field}"


def default_value(field: str):
    value = DEFAULTS[field]
    return int(value) if field in INT_INPUT_FIELDS else value


for field in INPUT_FIELDS:
    st.session_state.setdefault(field_key(field), default_value(field))


def reset_defaults():
    for field in INPUT_FIELDS:
        st.session_state[field_key(field)] = default_value(field)


def load_example(kind: str):
    reset_defaults()
    updates = HIGH_RISK_UPDATES if kind == "high" else LOW_RISK_UPDATES
    for field, value in updates.items():
        if field in INPUT_FIELDS:
            st.session_state[field_key(field)] = value


def current_values() -> dict:
    return {field: st.session_state[field_key(field)] for field in INPUT_FIELDS}


def full_input(values: dict) -> dict:
    """Combine the form values with typical values for background fields."""
    return {**DEFAULTS, **values}


def validation_errors(values: dict) -> list[str]:
    errors = []
    for semester, label in (("1st", "1"), ("2nd", "2")):
        enrolled = values[f"Curricular units {semester} sem (enrolled)"]
        approved = values[f"Curricular units {semester} sem (approved)"]
        if approved > enrolled:
            errors.append(
                f"Semester {label}: approved units cannot exceed enrolled units."
            )
    return errors


def render_field(field: str):
    label = pretty_name(field)
    key = field_key(field)
    help_text = FIELD_HELP.get(field)
    options = field_options(field)

    if options:
        values = [value for _, value in options]
        labels = {value: text for text, value in options}
        if len(options) <= 3:
            st.segmented_control(
                label,
                options=values,
                format_func=lambda value: labels[value],
                key=key,
                help=help_text,
                required=True,
                width="stretch",
            )
        else:
            st.selectbox(
                label,
                options=values,
                format_func=lambda value: labels[value],
                key=key,
                help=help_text,
            )
        return

    minimum, maximum, step = FIELD_RANGES.get(field, (0.0, None, 0.1))
    if field in INT_INPUT_FIELDS:
        st.number_input(
            label,
            min_value=int(minimum),
            max_value=int(maximum) if maximum is not None else None,
            step=int(step),
            key=key,
            help=help_text,
        )
    else:
        st.number_input(
            label,
            min_value=float(minimum),
            max_value=float(maximum) if maximum is not None else None,
            step=float(step),
            format="%.2f",
            key=key,
            help=help_text,
        )


def show_result(result: dict):
    probability = result["dropout_probability"]
    risk = result["risk_level"]

    with st.container(border=True):
        left, right = st.columns([2, 1])
        with left:
            st.metric(
                "Estimated dropout probability",
                f"{result['dropout_probability_percent']}%",
            )
            st.progress(min(max(probability, 0.0), 1.0))
        with right:
            st.metric("Risk level", risk)
            st.caption(
                "Flagged for review"
                if result["predicted_dropout"]
                else "Not flagged"
            )
        st.caption(
            f"Decision threshold: {result['threshold_used']:.2f}. "
            "The estimate uses the form fields plus typical values for "
            "background details that are not requested."
        )

    st.subheader("Key factors")
    reasons = pd.DataFrame(
        [
            {
                "Reason": item["reason"],
                "Area": item["category"],
                "Share": f"{item['percent_contribution']:.1f}%",
                "Direction": item["direction"],
            }
            for item in result["top_reasons"][:5]
        ]
    )
    st.dataframe(reasons, width="stretch", hide_index=True)
    st.caption(
        "Only the fields in the form are shown here. Background fields are held "
        "at typical values and are excluded from this list."
    )

    st.subheader("Where the estimate comes from")
    category_df = pd.DataFrame(result["category_contributions"])
    if not category_df.empty:
        figure = px.bar(
            category_df,
            x="percent_contribution",
            y="category",
            orientation="h",
            text="percent_contribution",
            color="percent_contribution",
            color_continuous_scale="Blues",
            labels={
                "percent_contribution": "Share of explanation (%)",
                "category": "",
            },
        )
        figure.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
        figure.update_layout(
            showlegend=False,
            coloraxis_showscale=False,
            height=300,
            margin=dict(l=10, r=40, t=20, b=20),
            xaxis_range=[0, max(60, category_df["percent_contribution"].max() * 1.2)],
        )
        st.plotly_chart(
            figure,
            width="stretch",
            alt="Bar chart of explanation share by category",
        )

    st.subheader("Recommended actions")
    for recommendation in result["recommendations"][:4]:
        st.markdown(f"- {recommendation}")

    st.caption(result["disclaimer"])


st.title("Student Dropout Risk Assistant")
st.markdown(
    "Enter the core details below to estimate dropout risk and receive "
    "supportive recommendations. This tool is designed for early support, "
    "not punishment."
)
st.info(
    "This prediction is a decision-support tool. Human review is required before "
    "any academic or financial action is taken.",
    title="Use with human review",
)

with st.container(horizontal=True):
    st.button("Use typical student", on_click=reset_defaults)
    st.button(
        "High-risk example",
        on_click=functools.partial(load_example, "high"),
    )
    st.button(
        "Low-risk example",
        on_click=functools.partial(load_example, "low"),
    )

with st.form("assessment_form"):
    for section, fields in APP_SECTIONS.items():
        st.markdown(f"#### {section}")
        columns = st.columns(2)
        for index, field in enumerate(fields):
            with columns[index % 2]:
                render_field(field)

    st.caption(
        f"Only these {len(INPUT_FIELDS)} fields are requested. Marital status, "
        "nationality, family occupation, application details, and macro "
        "indicators are not needed for the estimate; they use typical training "
        "values."
    )
    submitted = st.form_submit_button(
        "Predict risk", type="primary", width="stretch"
    )

if submitted:
    values = current_values()
    errors = validation_errors(values)
    if errors:
        for error in errors:
            st.error(error)
    else:
        try:
            st.session_state["result"] = predictor.predict(
                full_input(values),
                explanation_features=APP_EXPLANATION_FEATURES,
            )
        except Exception as exc:
            st.error("The prediction could not be generated.")
            st.exception(exc)

result = st.session_state.get("result")
if result:
    show_result(result)

st.divider()
st.subheader("Ask the assistant")
st.caption(
    "Commands: `predict`, `use typical student`, `explain risk`, "
    "`show recommendations`."
)

if "chat" not in st.session_state:
    st.session_state["chat"] = []

for message in st.session_state["chat"]:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

prompt = st.chat_input("Ask about dropout risk or type a command")
if prompt:
    st.session_state["chat"].append({"role": "user", "content": prompt})
    command = prompt.strip().lower()

    if command in {"predict", "predict risk"}:
        values = current_values()
        errors = validation_errors(values)
        if errors:
            reply = "Please fix the form first: " + " ".join(errors)
        else:
            result = predictor.predict(
                full_input(values),
                explanation_features=APP_EXPLANATION_FEATURES,
            )
            st.session_state["result"] = result
            reply = (
                f"Estimated dropout probability is "
                f"**{result['dropout_probability_percent']}%** "
                f"({result['risk_level']} risk)."
            )
    elif command in {"use typical student", "use default student", "default"}:
        reset_defaults()
        result = predictor.predict(
            dict(DEFAULTS),
            explanation_features=APP_EXPLANATION_FEATURES,
        )
        st.session_state["result"] = result
        reply = (
            "Loaded the typical student. "
            f"Estimated dropout probability is "
            f"**{result['dropout_probability_percent']}%** "
            f"({result['risk_level']} risk)."
        )
    elif command in {"explain risk", "explain"}:
        current = st.session_state.get("result")
        if current:
            lines = [
                f"- {item['reason']} ({item['category']}, "
                f"{item['percent_contribution']:.1f}%)"
                for item in current["top_reasons"][:5]
            ]
            reply = "Top factors:\n" + "\n".join(lines)
        else:
            reply = "Please run a prediction first."
    elif command in {"show recommendations", "recommendations"}:
        current = st.session_state.get("result")
        if current:
            reply = "Recommended actions:\n" + "\n".join(
                f"- {item}" for item in current["recommendations"]
            )
        else:
            reply = "Please run a prediction first."
    else:
        reply = (
            "I can help with: `predict`, `use typical student`, "
            "`explain risk`, `show recommendations`."
        )

    st.session_state["chat"].append({"role": "assistant", "content": reply})
    st.rerun()

with st.sidebar:
    st.header("Model information")
    st.metric("Decision threshold", f"{predictor.threshold:.2f}")
    st.metric("Form fields", len(INPUT_FIELDS))
    st.caption(
        f"The model uses {len(predictor.feature_columns)} internal features. "
        f"The other {len(HIDDEN_FIELDS)} background fields use typical values."
    )
    st.caption("Artifacts are loaded from models/. The app never retrains.")
    st.divider()
    st.markdown("**Risk bands**")
    for level, band in predictor.risk_bands.items():
        upper = min(band[1], 1.0)
        st.markdown(f"- {level}: {band[0]:.0%} - {upper:.0%}")
    st.divider()
    st.caption(
        "Privacy: do not enter names, student IDs, or other direct identifiers."
    )

In [ ]:
readme_text = "# Student Dropout Prediction and Explainable Assistant\n\nAn end-to-end, leakage-safe machine-learning project that estimates a student's\ndropout risk, explains the main factors behind the estimate, and suggests\nsupportive actions. It is built for **Kaggle first** and can be moved to a local\nStreamlit app later.\n\n> This is a decision-support tool, not an automatic decision. Human review is\n> required before any academic or financial action is taken.\n\n## What the project does\n\n1. Loads `dataset_outliers_removed_balanced.csv`.\n2. Creates a binary `is_dropout` target and optional three-class targets.\n3. Engineers semester-level academic features (approval rates, failed units,\n   effective grades, semester changes, risk flags, evaluation ratios).\n4. Fits frequency encoding and median imputation on the **training split only**.\n5. Trains an XGBoost classifier with early stopping on a validation split.\n6. Selects a decision threshold on validation data using Youden's J.\n7. Evaluates on a held-out test set (accuracy, precision, recall, F1, ROC AUC,\n   PR AUC, confusion matrix).\n8. Produces global SHAP importance, category-level contributions, and individual\n   explanations.\n9. Saves the model, preprocessor, feature list, threshold, category map, default\n   input values, and metadata.\n10. Provides a Kaggle ipywidgets assistant and a local Streamlit chatbot.\n\n## Dataset\n\nThe project expects this file:\n\n```text\ndataset_outliers_removed_balanced.csv\n```\n\nExpected columns include the 36 UCI student-performance features plus `Target`\nwith values `Graduate`, `Dropout`, or `Enrolled`.\n\nThe training notebook uses the full feature set. The app deliberately asks for\nonly 14 high-signal fields: course, age, gender, displaced status, admission\ngrade, semester 1/2 enrolled units, approved units, grades, tuition-fee status,\ndebt status, and scholarship status. Course names, female/male, and no/yes\nchoices are shown as text instead of numeric codes. Background details such as\nmarital status, nationality, family occupation, application mode, and macro\nindicators are filled with typical training values and are left out of the\nvisible key-factor list.\n\nThe file name says \"balanced\", but the version used here has:\n\n```text\nGraduate 2051\nDropout  1273\nEnrolled  747\n```\n\nThe binary target is therefore handled with `scale_pos_weight`; no SMOTE is\napplied. Outliers are not removed again.\n\n## Kaggle quickstart\n\n1. Create a Kaggle notebook.\n2. Add `dataset_outliers_removed_balanced.csv` as a Kaggle dataset (or upload it\n   to the notebook's working directory).\n3. Import/download `kaggle/Student_Dropout_Kaggle.ipynb` and open it in Kaggle.\n4. Click **Run all**.\n5. Use the 14-field ipywidgets form at the bottom to make predictions and ask\n   the assistant for explanations or recommendations.\n6. Download `/kaggle/working/student_dropout_project.zip` from the notebook output.\n\nThe notebook searches `/kaggle/input` for the CSV by filename, so the Kaggle\ndataset slug can be anything.\n\n## Local quickstart\n\nAfter downloading and unzipping the project:\n\n```bash\npip install -r requirements.txt\nstreamlit run app/chatbot_app.py\n```\n\nThe app loads the saved artifacts and never retrains the model. It asks for the\n14 core fields, uses readable labels for coded values, and fills omitted\nbackground fields with typical training values. It was tested with Streamlit\n1.66; `requirements.txt` uses 1.50+ because the app uses the current `width`\nAPI.\n\n## Project structure\n\n```text\nstudent_dropout_project/\n\u251c\u2500\u2500 data/\n\u2502   \u2514\u2500\u2500 dataset_outliers_removed_balanced.csv\n\u251c\u2500\u2500 models/\n\u2502   \u251c\u2500\u2500 dropout_model.joblib\n\u2502   \u251c\u2500\u2500 preprocessor.joblib\n\u2502   \u251c\u2500\u2500 feature_columns.json\n\u2502   \u251c\u2500\u2500 threshold.json\n\u2502   \u251c\u2500\u2500 metadata.json\n\u2502   \u251c\u2500\u2500 category_map.json\n\u2502   \u2514\u2500\u2500 default_input_values.json\n\u251c\u2500\u2500 outputs/\n\u2502   \u251c\u2500\u2500 evaluation_report.json\n\u2502   \u251c\u2500\u2500 feature_importance.csv\n\u2502   \u251c\u2500\u2500 shap_global_importance.csv\n\u2502   \u251c\u2500\u2500 category_contribution.csv\n\u2502   \u2514\u2500\u2500 example_prediction.json\n\u251c\u2500\u2500 src/\n\u2502   \u251c\u2500\u2500 __init__.py\n\u2502   \u251c\u2500\u2500 feature_engineering.py\n\u2502   \u251c\u2500\u2500 preprocessing.py\n\u2502   \u2514\u2500\u2500 predict.py\n\u251c\u2500\u2500 app/\n\u2502   \u2514\u2500\u2500 chatbot_app.py\n\u251c\u2500\u2500 README.md\n\u2514\u2500\u2500 requirements.txt\n```\n\n## Notebook flow\n\nThe Kaggle notebook is organized as numbered cells with a short explanation\nbefore each step:\n\n1. Setup, imports, dataset discovery.\n2. Load and inspect data.\n3. Define targets and feature groups.\n4. Feature engineering.\n5. Train/validation/test split.\n6. Preprocessing fitted on train only.\n7. XGBoost training with early stopping.\n8. Threshold selection on validation.\n9. Evaluation on validation and test.\n10. XGBoost feature importance.\n11. Global SHAP explanation and category contribution.\n12. Save model artifacts and default input values.\n13. Prediction, SHAP explanation, and recommendation module.\n14. Kaggle ipywidgets chatbot with the 14-field form.\n15. Export Streamlit app, README, requirements, and ZIP.\n\n## Example input\n\n```python\nstudent = {\n    \"Course\": 9500,                  # Nursing\n    \"Age at enrollment\": 20,\n    \"Gender\": 0,                     # Female\n    \"Displaced\": 0,                  # No\n    \"Admission grade\": 120.0,\n    \"Curricular units 1st sem (enrolled)\": 6,\n    \"Curricular units 1st sem (approved)\": 4,\n    \"Curricular units 1st sem (grade)\": 12.0,\n    \"Curricular units 2nd sem (enrolled)\": 6,\n    \"Curricular units 2nd sem (approved)\": 3,\n    \"Curricular units 2nd sem (grade)\": 11.0,\n    \"Tuition fees up to date\": 1,    # Yes\n    \"Debtor\": 0,                     # No\n    \"Scholarship holder\": 0,         # No\n}\n```\n\nAny omitted field is filled with a training median or mode. The app passes\ntypical values for the background fields it does not ask for and filters those\nfields out of the visible explanation.\n\n## Example output shape\n\n```json\n{\n  \"dropout_probability\": 0.78,\n  \"risk_level\": \"High\",\n  \"predicted_dropout\": 1,\n  \"top_reasons\": [\n    {\n      \"feature\": \"overall_approval_rate\",\n      \"category\": \"Engagement\",\n      \"value\": 0.22,\n      \"shap_value\": 0.18,\n      \"percent_contribution\": 21.5,\n      \"direction\": \"increases risk\",\n      \"reason\": \"Low overall approval rate (22%)\"\n    }\n  ],\n  \"category_contributions\": [\n    {\"category\": \"Engagement\", \"percent_contribution\": 46.2}\n  ],\n  \"recommendations\": [\n    \"Contact the student for a supportive check-in.\",\n    \"Create a study recovery plan.\"\n  ],\n  \"disclaimer\": \"This prediction is a decision-support tool...\"\n}\n```\n\n## Risk levels\n\n```text\nLow:      probability < 0.35\nMedium:   0.35 <= probability < 0.65\nHigh:     0.65 <= probability < 0.85\nCritical: probability >= 0.85\n```\n\nThe tuned threshold (selected on validation data) is used for the binary\n`predicted_dropout` flag. Risk levels use the fixed bands above.\n\n## Ethical use\n\n- Do not use this model as the sole basis for denying admission, scholarships,\n  or other opportunities.\n- Do not enter names, student IDs, or other direct identifiers.\n- Review explanations with a counselor or academic advisor.\n- Monitor for bias across demographic groups before any real deployment.\n- The model identifies patterns in historical data; it does not establish causes.\n\n## Reproducibility\n\n- `RANDOM_STATE = 42`\n- Stratified 70/15/15 split\n- Preprocessing fitted on training data only\n- Threshold tuned on validation data only\n- Test set used only for final evaluation\n"
requirements_text = "pandas>=2.0\nnumpy>=1.24\nscikit-learn>=1.3\nxgboost>=2.0\nshap>=0.44\njoblib>=1.3\nmatplotlib>=3.7\nseaborn>=0.13\nstreamlit>=1.50\nplotly>=5.18\nipywidgets>=8.0\n"

(PROJECT_DIR / 'README.md').write_text(readme_text, encoding='utf-8')
(PROJECT_DIR / 'requirements.txt').write_text(requirements_text, encoding='utf-8')

zip_path = (
    Path('/kaggle/working/student_dropout_project.zip')
    if Path('/kaggle/working').exists()
    else PROJECT_DIR.parent / 'student_dropout_project.zip'
)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in PROJECT_DIR.rglob('*'):
        if file.is_file() and '__pycache__' not in file.parts and file.suffix != '.pyc':
            archive.write(file, file.relative_to(PROJECT_DIR))

print('ZIP created:', zip_path)
print('\nProject files:')
for file in sorted(PROJECT_DIR.rglob('*')):
    if file.is_file() and '__pycache__' not in file.parts and file.suffix != '.pyc':
        print(' -', file.relative_to(PROJECT_DIR))

## Done

You now have:

- a trained and threshold-tuned XGBoost model;
- a leakage-safe preprocessing object;
- validation and test metrics;
- global and individual SHAP explanations;
- a working Kaggle risk assistant;
- a Streamlit app and the full project zip for the local shift.

Download `student_dropout_project.zip` from `/kaggle/working` to keep the model,
outputs, source modules, and app. When you move locally, unzip it and run:

```bash
pip install -r requirements.txt
streamlit run app/chatbot_app.py
```

Use this system for support, not punishment. Human review is required.